# 01 — A tiny character-level transformer

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/01_tiny_transformer.ipynb)

Companion to §2 of *From Brains to Models*.

**Why this notebook.** Large language models now draft clinical notes, pass licensing examinations, and hold fluent conversations, and it is tempting to read that fluency as understanding. This notebook removes the temptation by shrinking the machinery to a size one can hold in mind. A one-layer, four-head transformer, trained in seconds on a few hundred toy sentences, performs exactly the operation a frontier model performs — it estimates the probability of the next symbol given the symbols so far — and nothing else. Watching it learn makes three things concrete: what *next-token prediction* is, what *attention* actually computes, and what is absent from the whole procedure — no body, no perception, no cost for being wrong. Section 2 needs only this one fact, and it is easier to grant once seen than once asserted.

**Claim.** Foundation models implement next-token prediction with attention, trained on text from which embodiment has been removed.

**Method.** One-layer, four-head character-level transformer on a small designed corpus. Seed 7.

**Result.** Causal attention maps, falling next-character NLL, mass on object-initial letters.

**Limit.** Not a commercial LLM, not a brain, not a world model.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) and [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)).** The transformer is the network architecture behind current language models. Attention is its core operation: to predict the next symbol, the model looks back over the earlier symbols and forms a weighted summary of them. It is a learned "glance back", not a lookup of meaning.
- **Next-token (here, next-character) prediction.** The only task: given the text so far, output a probability for each possible next symbol. Fluency is a by-product of doing this well; there is no separate objective for "understanding".
- **Training loss (negative log-likelihood).** A score that is low when the model assigns high probability to the symbol that actually came next. Training only makes this score small on the corpus — much like measuring how surprised you are, on average, by the next value in a record.
- **Causal mask.** A rule forbidding each position from "seeing" symbols that come later, so prediction is always left-to-right, as in real generation.
- **[Foundation model](https://en.wikipedia.org/wiki/Foundation_model) / LLM.** The same objective and architecture, scaled to enormous corpora. This toy is that object with the size turned down.
- **[Embodiment](https://plato.stanford.edu/entries/embodied-cognition/) — what is missing.** The model is trained on text alone. There is no body, no sensor, and no consequence; a completion can be fluent without anyone having examined a patient.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **[Transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)).** Token and position embeddings, one or more layers of multi-head [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) and a position-wise feed-forward network, then a linear read-out to logits over the vocabulary $\mathcal{V}$.
- **[Attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)).** One head computes $\mathrm{softmax}\!\big(QK^\top/\sqrt{d}+M\big)V$, a data-dependent convex combination of value vectors; $M$ is the causal mask, with $M_{ij}=-\infty$ for $j>i$.
- **Objective.** Empirical-risk minimisation of the cross-entropy (negative log-likelihood) $\mathbb{E}\big[-\log p_\theta(x_{t+1}\mid x_{\le t})\big]$. A uniform guess gives $\ln|\mathcal{V}|$; here $|\mathcal{V}|=27$, so chance $\approx 3.30$ nats.
- **Proper scoring rule.** $-\log p$ is minimised, in the infinite-data limit, only by the true conditional $P(x_{t+1}\mid x_{\le t})$; a low value certifies calibration on this corpus, not comprehension.
- **[Foundation model](https://en.wikipedia.org/wiki/Foundation_model) and [in-context learning](https://en.wikipedia.org/wiki/In-context_learning).** The same $p_\theta(x_{t+1}\mid x_{\le t})$ at scale; new behaviours can appear from the prompt without weight updates, and remain token statistics.
- **[Embodiment](https://plato.stanford.edu/entries/embodied-cognition/).** The variable absent from every term above: no state the model must keep within viable bounds. §2 makes this the point of the comparison.

</details>

The same code writes `paper/figures/tiny_transformer_attention.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.tiny_transformer import (
    CharTokenizer,
    TinyTransformer,
    make_corpus,
    render_paper_figure,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)

repo root: <checkout>
seed: 7


## A small designed corpus

Foundation models train on a mixture of genres drawn from the web — stories, headlines, encyclopedia lines, FAQs — not on a single short text. Here that mixture is shrunk to five frames that still share the same nouns:

- story: `once upon a time the cat sat on the mat.`
- headline: `breaking from paris: the dog sat on the log.`
- wiki: `according to wikipedia the fox sat on the hat.`
- FAQ: `q: where did the cat sit? a: on the box.`
- and the bare line the figure uses: `the cat sat on the mat.`

Four hundred sentences, a few dozen types. The repeating sentence frame (`the cat sat on the …`) is what makes the attention maps readable. The genre wrappers are what make the corpus resemble a mixture of genres rather than a single template.

To keep the contrast in view: a frontier foundation model is trained on a corpus of web pages, books, code, dialogue, and reference works whose size is measured in *trillions* of tokens — many orders of magnitude beyond the few thousand characters here — and it reads that text as sub-word tokens rather than single characters. The toy keeps the *objective* identical (predict the next token given the ones before) while shrinking the *data* to something a reader can inspect line by line. The primer below defines the terms that recur once the model is built and trained.

In [2]:
from collections import Counter

text = make_corpus(seed=SEED)
tokenizer = CharTokenizer.from_text(text)
print(f"{len(text)} characters, vocab {tokenizer.vocab_size}: {tokenizer.chars!r}")

sentences = [s.strip() + "." for s in text.split(".") if s.strip()]
types = Counter(sentences)
print(f"{len(sentences)} sentences, but only {len(types)} unique types:\n")
for sent, n in types.most_common():
    print(f"  {n:3d}×  {sent}")

# Consecutive pairs are the smallest windows that still vary. Distance is
# 1 − SequenceMatcher ratio (order-aware; a repeated sentence is close to
# itself). Seed with the two farthest windows, then greedily add the one
# that maximises its min-distance to those already chosen.
from difflib import SequenceMatcher

windows = list(dict.fromkeys(
    f"{sentences[i]} {sentences[i + 1]}" for i in range(len(sentences) - 1)
))


def seq_dist(a: str, b: str) -> float:
    return 1.0 - SequenceMatcher(None, a, b).ratio()


slen = 5
best_d, seed_i, seed_j = -1.0, 0, 1
for i, a in enumerate(windows):
    for j, b in enumerate(windows[i + 1 :], start=i + 1):
        d = seq_dist(a, b)
        if d > best_d:
            best_d, seed_i, seed_j = d, i, j

selected = [windows[seed_i], windows[seed_j]]
remaining = [w for w in windows if w not in selected]
while remaining and len(selected) < slen:
    best = max(remaining, key=lambda w: min(seq_dist(w, s) for s in selected))
    selected.append(best)
    remaining.remove(best)

print(
    f"\n{len(selected)} most different two-sentence windows "
    f"(greedy max-min sequence distance; first two are the farthest pair):\n"
)
for i, window in enumerate(selected, 1):
    print(f"{i:3d}  {window}")


15744 characters, vocab 27: [' ', '.', ':', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y']
400 sentences, but only 79 unique types:

   15×  the fox sat on the box.
   12×  q: where did the fox sit? a: on the hat.
   10×  according to wikipedia the dog sat on the log.
    9×  once upon a time the dog sat on the box.
    9×  according to wikipedia the dog sat on the box.
    9×  the fox sat on the log.
    9×  once upon a time the cat sat on the mat.
    9×  according to wikipedia the cat sat on the mat.
    9×  according to wikipedia the fox sat on the hat.
    8×  the fox sat on the mat.
    8×  according to wikipedia the dog sat on the hat.
    8×  once upon a time the fox sat on the mat.
    7×  q: where did the cat sit? a: on the log.
    7×  according to wikipedia the fox sat on the log.
    7×  the dog sat on the hat.
    7×  the dog sat on the log.
    7×  q: where did the dog sit? a: on the box.
    7×  th


5 most different two-sentence windows (greedy max-min sequence distance; first two are the farthest pair):

  1  q: where did the cat sit? a: on the hat. q: where did the fox sit? a: on the mat.
  2  the dog sat on the mat. according to wikipedia the cat sat on the box.
  3  the cat sat on the mat. breaking from tokyo: the dog sat on the log.
  4  once upon a time the fox sat on the log. breaking from paris: the fox sat on the mat.
  5  q: where did the fox sit? a: on the box. once upon a time the fox sat on the hat.


<details>
<summary>How the “most different” windows are chosen</summary>

The corpus has 400 sentences but only 18 types, so “show diverse text” has to mean something more than unique sentences. The cell builds every <em>consecutive pair</em> (the smallest window that still varies), drops duplicates, and then picks a small subset that is spread out.

<strong>Distance.</strong> Two windows <em>A</em> and <em>B</em> are scored with <code>1 − SequenceMatcher(A, B).ratio()</code> from the Python standard library. That ratio is order-aware: it asks how much one string can be turned into the other by inserts, deletes, and replacements. A sentence repeated twice is close to itself. A bag-of-words score (Jaccard) is not used here — it treats short repeats as “far from everything,” which is how a list full of <code>the fox sat. the fox sat.</code> can appear.

<strong>Seed.</strong> Compare every pair of windows and keep the two with the largest distance. Those are the first two lines in the printed list.

<strong>Greedy max-min.</strong> While the list is shorter than <code>slen</code>, add the remaining window whose <em>nearest</em> already-chosen neighbour is as far away as possible. In other words: maximise the minimum distance to the set you already have.

<strong>What this is not.</strong> It is a standard farthest-point heuristic, not a proof that these are the unique globally optimal <code>slen</code> windows. Changing <code>slen</code> (or the distance) will change the set. The point for this notebook is only to see that the “corpus” is a handful of frames, not internet-scale text.

</details>

### Concepts used in this notebook

The rest of the notebook builds a transformer, trains it on the corpus above, and reads out its next-character predictions. That involves a compact set of terms shared by every foundation model. The two blocks below define them once — first in plain language, then in the notation used in the code — so the figures that follow can be read without a separate reference. Each block covers the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Training data (foundation models).** Web pages, books, code, dialogue, and reference works — [trillions of tokens](https://en.wikipedia.org/wiki/Byte_pair_encoding), learned only by predicting the next token, with no human labels. This notebook substitutes ~400 designed sentences so the same objective can be watched at a legible scale.
- **[Token](https://en.wikipedia.org/wiki/Byte_pair_encoding) and vocabulary.** A *token* is the smallest unit the model reads and predicts. Here it is a single character; in real models it is a *word-piece* (e.g. "hospital" might split into "hospit" + "al"). The *vocabulary* is the fixed list of all possible tokens — here 27 (the letters, space, and period). By analogy, tokens are like the graphemes an alphabet permits, and the vocabulary is the alphabet.
- **[Transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)).** The network trained here. It processes every position of a sequence at once and lets those positions exchange information through attention; it is the architecture behind current language models.
- **[Embedding](https://en.wikipedia.org/wiki/Word_embedding) and position embedding.** Each token is turned into a vector — a point in a high-dimensional space where related tokens sit close together (a learned code, loosely like a population code in cortex). Because the transformer sees all positions simultaneously and has no built-in sense of order, each *position* also gets a vector, the *position embedding*, added in to say where in the sequence a token sits.
- **[Attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) (multi-head).** The operation that lets each position look back over earlier positions and form a weighted summary of them, the weights set by relevance. *Multi-head* means several such look-backs run in parallel, each free to track a different relation (subject–verb agreement, the most recent noun, and so on).
- **Query, key, value.** The bookkeeping of attention, much like a content-addressable memory. Each position issues a *query* ("what am I looking for?"); each earlier position advertises a *key* ("what do I contain?"); the match between them sets the weight; and the *value* ("what I hand on") is what is actually summed into the result.
- **[Feed-forward network (FFN)](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) and [ReLU](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)).** After attention mixes information across positions, a small two-layer network refines each position on its own. Its nonlinearity is the *ReLU* (rectified linear unit): keep positive inputs, set negatives to zero — a threshold, like a neuron that responds only above baseline.
- **Linear readout and [softmax](https://en.wikipedia.org/wiki/Softmax_function).** A final linear layer maps each position's vector to one score per vocabulary token; *softmax* turns those scores into a probability distribution over the next token. Taking the most probable token, or sampling from the distribution, is how the model "writes".
- **[Likelihood and negative log-likelihood (NLL)](https://en.wikipedia.org/wiki/Cross-entropy).** The *likelihood* is the probability the model assigned to the text that actually occurred. The *NLL* is minus its logarithm — an average *surprise* that is small when the model expected what came next.
- **Loss.** The single number training reduces. Here it is the mean NLL over the corpus, in nats; a model that guessed uniformly among 27 tokens would score $\ln 27 \approx 3.30$.
- **Training ([backpropagation](https://en.wikipedia.org/wiki/Backpropagation)).** Begin from random weights; repeatedly show a batch of text, compute the loss, work out how each weight affected it (*backpropagation*), and nudge every weight a little to lower the loss (*gradient descent*). Repeat for many steps until the loss stops falling.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Corpus, [tokens](https://en.wikipedia.org/wiki/Byte_pair_encoding), vocabulary.** A sequence $x_1,\dots,x_T$ over a finite alphabet $\mathcal{V}$; here $|\mathcal{V}|=27$ characters. Production models use sub-word tokens (byte-pair encoding) with $|\mathcal{V}|\sim 10^4\text{–}10^5$ and $T$ in the trillions, fit by self-supervised next-token prediction (no labels).
- **[Embeddings](https://en.wikipedia.org/wiki/Word_embedding).** A token-embedding matrix $E\in\mathbb{R}^{|\mathcal{V}|\times d}$ and a position-embedding matrix $P\in\mathbb{R}^{L\times d}$ give the layer-0 input $h^{(0)}_t = E_{x_t} + P_t \in \mathbb{R}^{d}$; without $P$ the model would be permutation-invariant in $t$.
- **[Attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)).** Per head, $Q=hW_Q,\ K=hW_K,\ V=hW_V$ and $\mathrm{Attn}(h)=\mathrm{softmax}\!\big(QK^\top/\sqrt{d_k}+M\big)V$, with causal mask $M_{ij}=-\infty$ for $j>i$. Multi-head attention runs $H$ such maps in parallel, concatenates them, and projects by $W_O$; here one layer with $H=4$.
- **[FFN](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) and [ReLU](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)).** Position-wise, $\mathrm{FFN}(z)=W_2\,\mathrm{ReLU}(W_1 z + b_1)+b_2$ with $\mathrm{ReLU}(u)=\max(0,u)$. Each sublayer (attention, FFN) sits inside a residual connection with layer normalisation.
- **Linear readout, [softmax](https://en.wikipedia.org/wiki/Softmax_function).** Logits $\ell_t = h^{(L)}_t W_U \in \mathbb{R}^{|\mathcal{V}|}$ and $p_\theta(\cdot\mid x_{\le t}) = \mathrm{softmax}(\ell_t)$.
- **Likelihood, [cross-entropy](https://en.wikipedia.org/wiki/Cross-entropy) loss.** The sequence likelihood is $\prod_{t} p_\theta(x_{t+1}\mid x_{\le t})$. The loss is the mean negative log-likelihood $\mathcal{L}(\theta) = -\tfrac{1}{T}\sum_t \log p_\theta(x_{t+1}\mid x_{\le t})$ — the cross-entropy between the empirical next-token distribution and the model, in nats; chance $=\ln|\mathcal{V}|\approx 3.30$.
- **Training.** Minimise $\mathcal{L}$ by stochastic gradient descent on mini-batches; gradients $\nabla_\theta\mathcal{L}$ are computed by [backpropagation](https://en.wikipedia.org/wiki/Backpropagation) (reverse-mode automatic differentiation) and applied with the Adam optimiser. All runs use seed 7.

</details>

## One layer, four heads, next-character loss

Token + position embeddings, causal multi-head attention, a small ReLU FFN, then a linear readout. The training objective is next-character negative log-likelihood — predict what comes next.

In [3]:
model = TinyTransformer(vocab_size=tokenizer.vocab_size, seed=SEED)
history = model.train(tokenizer.encode(text), n_steps=500, seed=SEED)

print(model.generate("the cat sat on the ", tokenizer, n_tokens=50, temperature=0.6, seed=SEED))

step 0001/500  loss 3.2955


step 0050/500  loss 2.2944


step 0100/500  loss 1.4806


step 0150/500  loss 1.1291


step 0200/500  loss 0.8082


...

step 0300/500  loss 0.3894


step 0350/500  loss 0.3334


step 0400/500  loss 0.2847


step 0450/500  loss 0.2937


step 0500/500  loss 0.2428
the cat sat on the log. og once upon a time the dog sat on the log. o


## Paper figure

Three views of the same trained model. **A** is the internal view: how each attention head looks back over the prompt `the cat sat on the `. **B** is the training curve. **C** is the probability the model assigns to the next character.

How to read a heatmap in A. Each cell is a number between 0 and 1. The **row** is the query — the character that is “asking.” The **column** is the key — the character it is allowed to look at. Brighter means more weight. Because attention is *causal*, a character may only look at itself and what came before, so the upper triangle is black. The four heads are four different ways of asking the same question.

What to look for. Head 1 stays local (letters and spaces often attend to themselves or a recent copy). Head 2 reaches further back into earlier words. Heads 3 and 4 are peakier: one earlier character often takes almost all the mass. That is already “specialisation,” but it is crude — one layer, a tiny corpus.

Panel B should fall from about $\ln 27 \approx 3.3$ (chance, 27-character vocab: letters plus space, `.`, `:`, `?`) toward a value the model can almost recite. Panel C should put most mass on letters that start the objects the shared sentence frame uses (`h`/`l`/`m`/`b` for hat/log/mat/box), with leftover mass on animal-initial letters (`c`/`d`/`f`). The genre wrappers (breaking, wikipedia, q:) are how a corpus looks; the shared sentence frame is what the one-layer model actually locks onto.

In [4]:
fig = render_paper_figure(model, tokenizer, history)
plt.show()

out = write_figure(model, tokenizer, history, root / "paper" / "figures")
print("wrote", out)

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49831/2047507777.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/tiny_transformer_attention.pdf
wrote <checkout>/paper/figures/tiny_transformer_attention.png
wrote <checkout>/paper/figures/tiny_transformer_attention.pdf


## What to notice

**The objective is an LLM’s objective.** The model never sees “meaning.” It only scores $p(\text{next character}\mid\text{previous characters})$. After 500 steps the loss in B has dropped from chance ($\ln 27\approx 3.3$) to $\approx 0.24$, and the completion of `the cat sat on the ` is a sentence the corpus could have written.

**Attention is a weighted look-back, not a spotlight on concepts.** In A, pick the last row of any head — the space after the second `the`. That row is the model’s “where should I look before guessing the next letter?” Head 1 mostly looks at recent spaces; head 2 puts weight on the `e` of `the` and on `n`; heads 3–4 concentrate on a single earlier character. None of this is a picture of a cat. It is a routing pattern that happens to help next-character prediction on this formula.

**Heads specialise, but only a little.** With four heads the model can keep a local copy-ish look (head 1) and a longer look into the frame (head 2) at the same time. In a large language model the same idea is stacked over many layers. The maps get richer; the objective does not change.

**Panel C is next-*character* prediction, not next-word prediction.** The top bars `h`/`l`/`m`/`b` start `hat`/`log`/`mat`/`box` — the four objects that follow this sentence frame in every genre. Leftover mass on `c`/`d`/`f` is the model hedging toward `cat`/`dog`/`fox`. A tiny model has learned “an object-word is starting,” not a news story about a cat in Paris.

**Nothing here is embodied.** There is no body, no viability constraint, only a string. Scale the same objective and architecture — more layers, more data — and one has a foundation model: still next-token prediction, still attention, now over a corpus from which the body has been removed.

## Why this example is here

This notebook is not a tutorial on how to build ChatGPT. It is a controlled demonstration of the one fact §2 of the paper needs: **today’s large models are next-token predictors with attention, trained on text from which the body has been removed.** Everything else in the figure — four heads, a falling loss, a bar chart of letters — is there so that claim can be *seen*, not just asserted.

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never trained a language model. The two readings below are of the *same* object. Nothing in the code is medical, and nothing is a theorem about intelligence. The point is to make the training procedure concrete enough that the later contrast with brains is not a metaphor.

### What the model is asked to do

The alphabet here is 27 symbols: letters, space, and a few punctuation marks. A training sentence is a string $x_1 x_2 \ldots x_T$. At every position $t$ the model outputs a probability distribution over the next symbol,

$$
p_\theta(x_{t+1}\mid x_{\le t}),
$$

and is scored by the **negative log-likelihood** (cross-entropy)

$$
\mathcal{L} = -\log p_\theta(x_{t+1}\mid x_{\le t}).
$$

If the model is clueless, it spreads mass uniformly over the 27 symbols and $\mathcal{L}=\ln 27\approx 3.30$. After a few hundred steps on this toy corpus the loss in panel B is about $0.24$: the model has become very confident about what typically follows strings it has seen.

That is the whole objective. There is no extra loss for “understanding a cat,” “knowing Paris,” or “answering a question.” Fluency, if it appears, is a side-effect of being good at this score.

**For the clinician and the neuroscientist.** This is closer to *what finding usually comes next in records like these* than to *what is wrong with the patient*. A language model trained on notes can complete “the patient was started on …” with a plausible drug because that string is common, not because it has examined anyone. Diagnosis in a person is a different act: it is constrained by a body, by what would falsify the story, and by what happens if you are wrong. The model has only the string.

**For the mathematically inclined.** $\{p_\theta(\,\cdot\mid x_{\le t})\}$ is a family of categorical distributions on a finite set $\mathcal{V}$, parameterised by $\theta$ (the embeddings, the attention weights, the small feed-forward map). Training is empirical-risk minimisation of $\mathbb{E}[-\log p_\theta(X_{t+1}\mid X_{\le t})]$ on a fixed corpus. No semantics is assumed. A “word” is a substring that happens to recur; a “fact” is a statistical regularity.

### Why a small designed corpus, not a single short text

Foundation models are not trained on a single genre. A web-scale corpus mixes stories, headlines, encyclopedia lines, and question–answer pages. This notebook shrinks that mix to five frames that still share a sentence frame,

`the {animal} sat on the {object}`,

wrapped as a bare sentence, a fairy-tale opener, a dateline, a wiki attribution, and a FAQ. The nouns (`cat`/`dog`/`fox`, `mat`/`hat`/`log`/`box`) and the place-names in the headline (`paris`/`oslo`/`tokyo`) are placeholders that make the *mixture of genres* visible. After the prompt `the cat sat on the `, the next character is still an object-initial letter, not a city: the places live only in the opener.

The repeating sentence frame is what makes the attention maps in panel A readable. The genre wrappers are what keep the corpus from being a single template. Four hundred sentences and a few dozen word types are enough to train in seconds, and small enough that one can *see* what was learned. They are not a claim that the internet looks like this.

**For the clinician and the neuroscientist.** Imagine the same lab pattern appearing in a discharge summary, a news lede, a textbook sentence, and a patient FAQ. A model trained on all four will complete the pattern in any wrapper. That is not four kinds of clinical knowledge. It is one shared sentence frame in four wrappers. You already know the difference: the note is about a person; the FAQ page is not.

**For the mathematically inclined.** The data distribution is a mixture $P=\sum_g \pi_g P_g$ over a handful of templates $g$. The shared sentence frame is a long common substring; the wrappers are prefix/suffix variation. Attention can latch onto the shared sentence frame because it is a stable predictor of the next character across mixture components. That is a statement about $P$, not about reference to cats.

### How to read the three panels

**Panel A — attention is a weighted look-back.**  
Each head computes, for every character in the prompt, a set of weights over *earlier* characters (and itself). The row is the character that is asking (“query”); the column is the character it may look at (“key”). The black upper triangle is the **causal mask**: position $t$ is forbidden from seeing $t{+}1,t{+}2,\ldots$. After the softmax, each row is a probability distribution over the past. The next layer (here, a tiny feed-forward net) sees a *weighted average* of past representations, not a pointer to a concept.

Four heads are four such look-backs in parallel. In the figure they already differ: some stay local (nearby spaces and letters), others put mass on an earlier character in `the cat sat on the`. That is specialisation of *routing*, not four pictures of a cat.

**For the clinician and the neuroscientist.** When you write the next word of a note, you glance back — sometimes at the last few words (syntax), sometimes at an earlier noun (who), sometimes at a date. Multi-head attention is a crude, learned version of those glances. It is not a differential diagnosis, and it has no access to the patient, only to the prompt.

**For the mathematically inclined.** If $Q,K,V$ are linear maps of the token embeddings, one head is

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\big((QK^\top)/\sqrt{d}+M\big)\,V,
$$

where $M_{ij}=-\infty$ for $j>i$ (the causal mask) and each row of the softmax is a point in the simplex $\Delta^{t}$. A head is a data-dependent convex combination of past value vectors. Stacking heads is a direct sum of such maps. There is no constraint that a head align with a linguistic category.

**Panel B — the loss is a proper scoring rule.**  
The curve is mean $-\log p_\theta$ on random windows of the corpus. It starts near chance and falls as the shared sentence frame is learned. A lower number means: on strings like these, the model assigns more probability to the character that actually came next.

**For the clinician and the neuroscientist.** Think of a well-calibrated prediction of the next lab value, scored by how surprised you are when the result arrives. A model can be very good at that score on yesterday’s records and still be unsafe at the bedside — the score never saw the person, and it never paid the cost of a miss.

**For the mathematically inclined.** $-\log p$ is (up to a constant) the cross-entropy between the empirical next-symbol distribution and $p_\theta$. It is a strictly proper scoring rule: the unique minimiser in the infinite-data limit is the true conditional $P(X_{t+1}\mid X_{\le t})$. Finite data plus a tiny architecture yield an approximation, not the truth. The drop from $\ln|\mathcal{V}|$ to $\approx 0.24$ only says that this $P$ is highly predictable.

**Panel C — next *character*, not next thought.**  
The prompt is `the cat sat on the `. The bars are $p_\theta(\,\cdot\mid\text{prompt})$ over the 27-symbol alphabet. Mass concentrates on `h`/`l`/`m`/`b` because those start `hat`/`log`/`mat`/`box` — the four objects the corpus actually uses. Leftover mass on `c`/`d`/`f` is a hedge toward `cat`/`dog`/`fox`. The model has learned “an object-word is starting,” not a news story about a cat in Paris.

**For the clinician and the neuroscientist.** Autocomplete of “sat on the …” is not knowledge that someone sat. If a clinical LLM completes “start heparin if …” fluently, the same caveat applies: fluency is concentrated probability on a string, not an indication.

**For the mathematically inclined.** Panel C is one categorical vector in $\Delta^{26}$. The interesting event is that the support has collapsed onto a semantically meaningful subset of $\mathcal{V}$ *without* that subset being labelled in the loss. The label is in *our* reading of the bars.

### What the heads are not

They are not neurons with a preferred concept. They are not a parse tree, a knowledge graph, or a world model. In a large language model the same mechanism is stacked over many layers and tens of thousands of tokens; the maps get richer; **the objective does not change**. Scaling is the same empirical risk on a larger $\theta$ and a larger corpus — still $p(\text{next token}\mid\text{previous tokens})$.

### What this has to do with being human

The paper’s comparison is not “brains are transformers” and not “transformers are stupid.” It is that this training procedure is **disembodied**. The training signal is other people’s text and images. There is no body that must balance, no interoception, no metabolic cost, no social cost for being wrong about a particular person. A foundation model can sound like a clinician or a mathematician because clinicians and mathematicians wrote the corpus. It does not thereby *have* a clinic or a proof.

That is why the toy is worth studying. If one can see, in one layer and 27 characters, that competence at next-symbol prediction is not the same as having a world, then the rest of the paper — predictive coding, active inference, the gap between a score and a life — has a firm starting point.

If you change `SEED` or the frames and re-run, the maps and the top letters will move. The claim should not: **same objective, same look-back, no body.**


### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. If a radiologist and this model both complete “the patient was started on …”, what would have to be the same for that to count as the same [computational theory](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) (what is computed, and why)? What would still differ at implementation?

2. The loss is $-\log p(\text{next}\mid\text{past})$. Name a clinical prediction that is scored the same way. What cost does the clinical miss pay that this loss does not?

3. [Attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) is a weighted look-back over the prompt. When you glance back in a note, what extra constraint does that glance have that a causal head does not?

4. Four heads specialise in routing. What would it take to treat a head as a neurological “module,” and why is that a category error on this teaching model?

5. The corpus is a designed mixture of five frames. If a [foundation model](https://en.wikipedia.org/wiki/Foundation_model) sounds like a colleague, which mixture component is it matching — and what does that match fail to include?

6. Dropping the causal mask lets a query see characters that have not been written. Why is that fatal at generation time? Why is a speaker who plans a sentence not relying on the same shortcut?

7. Temperature sharpens or flattens $p(\cdot\mid\text{prompt})$. Is a more peaked categorical a stronger belief? What would a Bayesian clinician mean by belief that this vector does not?

8. Scaling keeps the same objective. If competence rises, which of Marr’s three rows has changed, and which has not?

9. [Embodiment](https://plato.stanford.edu/entries/embodied-cognition/) has been removed from the training signal. Name one measurement you could add, in principle, that would put a viability constraint back — and say why next-token mass is not that measurement.

10. Suppose the model’s next character is correct and the clinician’s next word is also correct. What, exactly, is still missing if the comparison is meant to be about being human?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Vaswani, A., et al. (2017).** [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762). *NeurIPS* 30. arXiv:1706.03762.
   - *(i) From brains to models.* The architecture of this notebook: a [transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) that replaces recurrence with [scaled dot-product attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) and a causal decoder. The paper’s translation numbers (28.4 BLEU EN–DE; 41.8 EN–FR) are not the claim here. The claim is the mechanism — query, key, value, and a mask that forbids the future.
   - *(ii) What AI reveals about being human.* Nothing in the architecture adds a body. [Attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) is a weighted look-back over tokens, not a sensorimotor loop. Scale the same objective and embodiment is still absent.

2. **Brown, T. B., et al. (2020).** [*Language Models are Few-Shot Learners*](https://arxiv.org/abs/2005.14165). *NeurIPS* 33. arXiv:2005.14165.
   - *(i) From brains to models.* Shows that the same next-token objective, at scale, yields [in-context learning](https://en.wikipedia.org/wiki/In-context_learning): a new format or rule can be picked up from examples in the prompt without a weight update. This notebook is that objective at toy scale, before few-shot competence appears.
   - *(ii) What AI reveals about being human.* In-context learning is still $p(\text{next}\mid\text{past})$. A temporary program in the context window is not a goal the system has, and not a world it must survive.

3. **Bommasani, R., et al. (2021).** [*On the Opportunities and Risks of Foundation Models*](https://arxiv.org/abs/2108.07258). arXiv:2108.07258.
   - *(i) From brains to models.* Defines the [foundation model](https://en.wikipedia.org/wiki/Foundation_model): trained on broad data at scale, adapted to many tasks. Figure 1 of this notebook is the same training procedure with the corpus and the width reduced until it fits a laptop.
   - *(ii) What AI reveals about being human.* The report’s adaptation story (fine-tune, prompt, tool use) is how agency is added after training. Biological development runs in the opposite order: a viable body first, language later.

4. **Elhage, N., et al. (2021).** [*A Mathematical Framework for Transformer Circuits*](https://transformer-circuits.pub/2021/framework/index.html). *Transformer Circuits Thread*.
   - *(i) From brains to models.* [Mechanistic interpretability](https://en.wikipedia.org/wiki/Mechanistic_interpretability) of a transformer as a sum of attention and MLP circuits writing into a residual stream. The attention maps in panel A are the toy of that picture: one layer, four heads, a causal mask.
   - *(ii) What AI reveals about being human.* A readable circuit is not a mind. Reverse-engineering the residual stream answers how a next token was computed. It does not add experience or a value.

5. **Jain, S., and Wallace, B. C. (2019).** [*Attention is not Explanation*](https://aclanthology.org/N19-1357/). *NAACL* 2019, 3543–3556.
   - *(i) From brains to models.* Learned [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) weights are often uncorrelated with gradient-based feature importance, and very different maps can yield the same prediction. Panel A is a mechanism, not a rationale. Do not read a bright cell as “the model attended because…”
   - *(ii) What AI reveals about being human.* The human temptation is to treat a heatmap as a reason. That is a projection of our report practices onto a weight matrix. The remainder is that we *give* reasons; the map only shows mass.

6. **Schrimpf, M., et al. (2021).** [*The neural architecture of language: Integrative modeling converges on predictive processing*](https://www.pnas.org/doi/10.1073/pnas.2105646118). *PNAS* 118(45), e2105646118.
   - *(i) From brains to models.* Transformer language models predict nearly all explainable variance in fMRI and ECoG responses to sentences; neural and behavioural fit track next-word accuracy, not other language tasks. A shared [predictive](https://en.wikipedia.org/wiki/Predictive_coding) objective is what the comparison licenses — not a shared implementation.
   - *(ii) What AI reveals about being human.* High brain-score is a representational match under a designed stimulus set. It is not evidence that the listener is a transformer, or that the model has a body that must keep predicting in order to live.

7. **Goldstein, A., et al. (2022).** [*Shared computational principles for language processing in humans and deep language models*](https://doi.org/10.1038/s41593-022-01026-4). *Nature Neuroscience* 25, 369–380.
   - *(i) From brains to models.* Nine ECoG listeners and an autoregressive language model, same narrative: both show next-word prediction before onset, post-onset surprise, and contextual embeddings. The opened paper states those three principles and denies that DLMs have the hallmarks of human cognition.
   - *(ii) What AI reveals about being human.* Shared next-word statistics are the convergence. The remainder is everything the listener still has and the model does not: a body, a context, and a life that continues when the tokens stop.

8. **Lindsey, J., Gurnee, W., Ameisen, E., et al. (2025).** [*On the Biology of a Large Language Model*](https://transformer-circuits.pub/2025/attribution-graphs/biology.html). *Transformer Circuits Thread*.
   - *(i) From brains to models.* Attribution graphs on Claude 3.5 Haiku: features and circuits for multi-hop lookup, rhyme planning, and hallucination. The authors treat the graphs as testable hypotheses, not proofs, and note that the replacement model captures the original only incompletely. This is [mechanistic interpretability](https://en.wikipedia.org/wiki/Mechanistic_interpretability) at production scale of the same architecture as panel A.
   - *(ii) What AI reveals about being human.* The title’s “biology” is a metaphor for circuit anatomy. The opened paper does not claim the model is an organism. Theme (ii) is exactly that gap: a traced circuit is not a living predictor.

9. **Ouyang, L., et al. (2022).** [*Training language models to follow instructions with human feedback*](https://arxiv.org/abs/2203.02155). *NeurIPS* 35. arXiv:2203.02155.
   - *(i) From brains to models.* Instruction tuning plus preference optimisation turns the next-token predictor into a conversational interface. Apparent agency in a chat window is that interface on this notebook’s objective.
   - *(ii) What AI reveals about being human.* Human feedback steers $p(\text{next}\mid\text{past})$. It does not install a preferred observation in a world the agent must survive. Preference is still a token statistic.


### Exercises

1. **Drop the causal mask.** The black upper triangle in panel A is not decoration: position $t$ is forbidden from seeing $t{+}1,t{+}2,\ldots$. Recompute the attention weights from the trained $Q$ and $K$ *without* that mask. How much mass does a mid-prompt query put on characters that have not been written yet? What would go wrong at generation time if the model had been allowed to learn that look-ahead?

<!-- solution-box: 01_1_drop_the_causal_mask.py -->
<details>
<summary><strong>Show solution</strong> — Drop the causal mask</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/01_1_drop_the_causal_mask.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.tiny_transformer import _softmax, train_demo
model, tokenizer, text, _ = train_demo(seed=SEED, n_steps=200)
prompt = "the cat sat on the "
ids = tokenizer.encode(prompt)
logits, causal = model.forward(ids, record_attn=True)
q, k = model._cache["q"], model._cache["k"]
scale = np.sqrt(model._d_head)
scores = np.einsum("bhtd,bhsd->bhts", q, k) / scale
leaky = _softmax(scores, axis=-1)

t = len(prompt)
# Query at the space after "cat" (index of that space).
q_at = prompt.index(" ", 4)
future = slice(q_at + 1, t)
leaked = float(leaky[0, :, q_at, future].sum() / model.n_heads)
blocked = float(causal[0, :, q_at, future].sum() / model.n_heads)
print(f"query at index {q_at} ({prompt[q_at]!r})")
print(f"mean mass on later characters, causal mask on : {blocked:.3f}")
print(f"mean mass on later characters, mask dropped   : {leaked:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4))
labels = [("␣" if c == " " else c) for c in prompt]
for ax, mat, title in (
    (axes[0], causal[0, 0, :t, :t], "head 1, causal"),
    (axes[1], leaky[0, 0, :t, :t], "head 1, no mask"),
):
    ax.imshow(mat, cmap="magma", vmin=0, vmax=max(0.2, float(mat.max())))
    ax.set_xticks(range(t), labels, rotation=90, fontsize=7)
    ax.set_yticks(range(t), labels, fontsize=7)
    ax.set_title(title, fontsize=9)
fig.suptitle("Without the mask, a query can look at characters that do not yet exist")
plt.tight_layout()
plt.show()
print(
    "Generation writes one character at a time. A look-ahead weight is a weight "
    "on a blank. Training with the mask dropped would let the loss cheat on the "
    "training window (the future tokens are already in the batch) and would not "
    "transfer to left-to-right sampling. The mask is what makes next-token "
    "prediction a well-posed online objective, not a better picture of a cat."
)
```

</details>
<!-- /solution-box -->


2. **A unigram is not enough.** Panel B falls well below $\ln|\mathcal{{V}}|$. How much of that drop is just “some letters are common”? Fit a unigram and a bigram on the same designed corpus and compare their next-character NLL on held-out windows to the trained transformer. What does the remaining gap say about what attention is for here?

<!-- solution-box: 01_2_a_unigram_is_not_enough.py -->
<details>
<summary><strong>Show solution</strong> — A unigram is not enough</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/01_2_a_unigram_is_not_enough.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.tiny_transformer import CharTokenizer, make_corpus, train_demo
model, tokenizer, text, _ = train_demo(seed=SEED, n_steps=200)
ids = tokenizer.encode(text)
n = len(ids)
cut = int(0.85 * n)
train_ids, test_ids = ids[:cut], ids[cut:]

unigram = np.ones(tokenizer.vocab_size)
for i in train_ids:
    unigram[int(i)] += 1.0
unigram = unigram / unigram.sum()

bigram = np.ones((tokenizer.vocab_size, tokenizer.vocab_size))
for a, b in zip(train_ids[:-1], train_ids[1:]):
    bigram[int(a), int(b)] += 1.0
bigram = bigram / bigram.sum(axis=1, keepdims=True)

def nll_unigram(seq):
    return float(-np.mean(np.log(unigram[seq])))

def nll_bigram(seq):
    return float(-np.mean(np.log(bigram[seq[:-1], seq[1:]])))

def nll_model(seq, block=32):
    losses = []
    for start in range(0, len(seq) - block - 1, block):
        ctx = seq[start : start + block]
        tgt = seq[start + 1 : start + 1 + block]
        logits, _ = model.forward(ctx)
        p = np.exp(logits[0] - logits[0].max(axis=-1, keepdims=True))
        p = p / p.sum(axis=-1, keepdims=True)
        losses.append(-np.log(np.clip(p[np.arange(block), tgt], 1e-9, 1.0)).mean())
    return float(np.mean(losses))

print(f"chance (ln V)   {np.log(tokenizer.vocab_size):.3f}")
print(f"unigram  test   {nll_unigram(test_ids):.3f}")
print(f"bigram   test   {nll_bigram(test_ids):.3f}")
print(f"1-layer  test   {nll_model(test_ids):.3f}")
print(
    "\nThe unigram knows that space and e are common. The bigram knows that "
    "'t' often follows 'a' in 'cat' / 'sat' / 'mat'. The transformer still "
    "wins: the shared sentence frame is longer than one previous character. "
    "Attention is a look-back over a window, not a better unigram. It is still "
    "not a world model."
)
```

</details>
<!-- /solution-box -->


3. **Temperature is not a belief.** `generate` has a `temperature` argument. Sample 20 continuations of `the cat sat on the ` at $T=0$ (argmax), $T=0.4$, and $T=1.2$. How does the set of object-words change? Why is a sharper sample not a more committed belief about a cat?

<!-- solution-box: 01_3_temperature_is_not_a_belief.py -->
<details>
<summary><strong>Show solution</strong> — Temperature is not a belief</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/01_3_temperature_is_not_a_belief.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.tiny_transformer import train_demo
model, tokenizer, _, _ = train_demo(seed=SEED, n_steps=400)
prompt = "the cat sat on the "
temps = (0.0, 0.4, 1.2)
objects = ("mat", "hat", "log", "box")

for T in temps:
    texts = [
        model.generate(prompt, tokenizer, n_tokens=12, temperature=T, seed=SEED + k)
        for k in range(20)
    ]
    tails = [t[len(prompt) :].split()[0] if t[len(prompt) :].split() else "" for t in texts]
    hits = {w: sum(w in tail for tail in tails) for w in objects}
    uniq = len(set(tails))
    print(f"T={T:<4}  unique first-words {uniq:2d}  object hits {hits}")
    print("   e.g.", texts[0][len(prompt) : 28])

print(
    "\nT=0 repeats the mode. Raising T spreads mass; more object-words appear, "
    "and so do broken spellings. Temperature rescales a categorical. It does not "
    "add a world that could make one completion true of a particular cat. A "
    "confident sample is a peaked next-character distribution, not a belief."
)
```

</details>
<!-- /solution-box -->
